In [10]:
import pandas as pd
from sklearn.preprocessing import OrdinalEncoder,LabelEncoder,OneHotEncoder

## Binary Encoding
Teknik ini menggabungkan manfaat Pengkodean One Hot dan pengkodean label.dengan motode ini dapat mengurangi dimensi (colom encoding) tanpa
mengurangi atau merusak informasi pada data

### 1. Memuat Dataset

In [12]:
df = pd.read_csv('../0.Dataset/employee_feedback.csv')
print(df.head())

  buying  maint doors persons lug_boot safety  class
0  vhigh  vhigh     2       2    small    low  unacc
1  vhigh  vhigh     2       2    small    med  unacc
2  vhigh  vhigh     2       2    small   high  unacc
3  vhigh  vhigh     2       2      med    low  unacc
4  vhigh  vhigh     2       2      med    med  unacc


### 2. Pengkodean Label

In [14]:
label = LabelEncoder()
df['class_encoded'] = label.fit_transform(df['class'])

print(f"Class labels Mapping:{dict(zip(label.classes_,label.transform(label.classes_)))}")
print(f"output label encoded:\n {df[['class','class_encoded']].head()}")

Class labels Mapping:{'acc': np.int64(0), 'good': np.int64(1), 'unacc': np.int64(2), 'vgood': np.int64(3)}
output label encoded:
    class  class_encoded
0  unacc              2
1  unacc              2
2  unacc              2
3  unacc              2
4  unacc              2


### 3. Pengkodean One-Hot

In [20]:
categorical_cols = ['buying','maint','doors','persons','lug_boot','safety']
onehot = OneHotEncoder(sparse_output=False)

onehot_array = onehot.fit_transform(df[categorical_cols])
print(f"One-Hot feature names: {onehot.get_feature_names_out(categorical_cols)}")

onehot_df = pd.DataFrame(onehot_array,columns=onehot.get_feature_names_out(categorical_cols))
df_concat = pd.concat([df.reset_index(drop=True),onehot_df],axis=1)

df_concat.head()

One-Hot feature names: ['buying_high' 'buying_low' 'buying_med' 'buying_vhigh' 'maint_high'
 'maint_low' 'maint_med' 'maint_vhigh' 'doors_2' 'doors_3' 'doors_4'
 'doors_5more' 'persons_2' 'persons_4' 'persons_more' 'lug_boot_big'
 'lug_boot_med' 'lug_boot_small' 'safety_high' 'safety_low' 'safety_med']


,buying,maint,doors,persons,lug_boot,safety,class,class_encoded,buying_high,buying_low,...,doors_5more,persons_2,persons_4,persons_more,lug_boot_big,lug_boot_med,lug_boot_small,safety_high,safety_low,safety_med
0,vhigh,vhigh,2,2,small,low,unacc,2,0.0,0.0,...,0.0,1.0,0.0,0.0,0.0,0.0,1.0,0.0,1.0,0.0
1,vhigh,vhigh,2,2,small,med,unacc,2,0.0,0.0,...,0.0,1.0,0.0,0.0,0.0,0.0,1.0,0.0,0.0,1.0
2,vhigh,vhigh,2,2,small,high,unacc,2,0.0,0.0,...,0.0,1.0,0.0,0.0,0.0,0.0,1.0,1.0,0.0,0.0
3,vhigh,vhigh,2,2,med,low,unacc,2,0.0,0.0,...,0.0,1.0,0.0,0.0,0.0,1.0,0.0,0.0,1.0,0.0
4,vhigh,vhigh,2,2,med,med,unacc,2,0.0,0.0,...,0.0,1.0,0.0,0.0,0.0,1.0,0.0,0.0,0.0,1.0


### 4. Pengkodean Ordinal

In [23]:
ordinals_cols = ['safety']
categories_order = [['low','med','high']]

ordinal = OrdinalEncoder(categories=categories_order)
df['safety_ordinal'] = ordinal.fit_transform(df[['safety']])

print(f"Oridnal Encode:\n {df[['safety','safety_ordinal']].head()}")

Oridnal Encode:
   safety  safety_ordinal
0    low             0.0
1    med             1.0
2   high             2.0
3    low             0.0
4    med             1.0


### 5. Menggabungkan Data dengan ColumnTransformer

In [24]:
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline

In [27]:
ordinal_feature = ['safety']
ordinal_categories = [['low','med','high']]
nominal_features = ['buying', 'maint', 'doors', 'persons', 'lug_boot']

preprocessor = ColumnTransformer(
    transformers=[
        ('ordinal',OrdinalEncoder(categories=ordinal_categories),ordinal_feature),
        ('nominal',OneHotEncoder(sparse_output=False),nominal_features)
    ]
)

features = ordinal_feature + nominal_features
X  =df[features]
X_prepared = preprocessor.fit_transform(X)

print(f"Transformed shape: {X_prepared.shape}")

Transformed shape: (1728, 19)


### 6. Inspeksi dan Kumpulan Data Hasil

In [35]:
import numpy as np
final_df = pd.DataFrame(np.hstack([X_prepared,df[['class_encoded']].values]), columns= list(preprocessor.get_feature_names_out()) + ['class_encoded'])

final_df


,ordinal__safety,nominal__buying_high,nominal__buying_low,nominal__buying_med,nominal__buying_vhigh,nominal__maint_high,nominal__maint_low,nominal__maint_med,nominal__maint_vhigh,nominal__doors_2,nominal__doors_3,nominal__doors_4,nominal__doors_5more,nominal__persons_2,nominal__persons_4,nominal__persons_more,nominal__lug_boot_big,nominal__lug_boot_med,nominal__lug_boot_small,class_encoded
0,0.0,0.0,0.0,0.0,1.0,0.0,0.0,0.0,1.0,1.0,0.0,0.0,0.0,1.0,0.0,0.0,0.0,0.0,1.0,2.0
1,1.0,0.0,0.0,0.0,1.0,0.0,0.0,0.0,1.0,1.0,0.0,0.0,0.0,1.0,0.0,0.0,0.0,0.0,1.0,2.0
2,2.0,0.0,0.0,0.0,1.0,0.0,0.0,0.0,1.0,1.0,0.0,0.0,0.0,1.0,0.0,0.0,0.0,0.0,1.0,2.0
3,0.0,0.0,0.0,0.0,1.0,0.0,0.0,0.0,1.0,1.0,0.0,0.0,0.0,1.0,0.0,0.0,0.0,1.0,0.0,2.0
4,1.0,0.0,0.0,0.0,1.0,0.0,0.0,0.0,1.0,1.0,0.0,0.0,0.0,1.0,0.0,0.0,0.0,1.0,0.0,2.0
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
1723,1.0,0.0,1.0,0.0,0.0,0.0,1.0,0.0,0.0,0.0,0.0,0.0,1.0,0.0,0.0,1.0,0.0,1.0,0.0,1.0
1724,2.0,0.0,1.0,0.0,0.0,0.0,1.0,0.0,0.0,0.0,0.0,0.0,1.0,0.0,0.0,1.0,0.0,1.0,0.0,3.0
1725,0.0,0.0,1.0,0.0,0.0,0.0,1.0,0.0,0.0,0.0,0.0,0.0,1.0,0.0,0.0,1.0,1.0,0.0,0.0,2.0
1726,1.0,0.0,1.0,0.0,0.0,0.0,1.0,0.0,0.0,0.0,0.0,0.0,1.0,0.0,0.0,1.0,1.0,0.0,0.0,1.0
